# T5：MERFISH 小鼠下丘脑三次独立留出切片验证

依次留出 `merfish_1`、`merfish_2`、`merfish_3`，每次用另外四张切片从头训练 STADiffuser，再在目标细胞的已知坐标上生成表达。

- 直接读取原始五个 h5ad，不要求先转换；也可读一个合并 h5ad。
- 留出表达不参加训练、潜空间归一化、图编码或条件推断。目标真实注释也不作为生成条件。
- 默认保留条件扩散：从训练切片的空间邻居推断目标 `annotation`，这里 annotation 是**解剖区域**。这是针对留出任务的适配协议。
- XY 沿用源文件坐标，Z 来自数值 slice_id。需输入已处于可比较的共同 XY 坐标系，本 Notebook 不做切片配准。
- 按 2026.10.18 周报报告平均基因表达 MSE/MAE/PCC、稀有类同类空间覆盖召回；只评价 STADiffuser。
- 默认完整预算为预训练 200 轮、对齐 300 轮、扩散训练 500 轮、生成 1000 步；每折三次生成取平均。首次可开启 SMOKE_TEST 检查流程。

本地只检查/转换数据时，运行到“划分检查”即可，不需要导入模型或启动训练。完整说明见项目中的 `experiments/MERFISH_HOLDOUT.md`。


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import anndata as ad
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "stadiffuser" / "pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 STADiffuser 项目或其子目录打开 Notebook")
sys.path.insert(0, str(PROJECT_ROOT))
from experiments import merfish_holdout as mh
print("Project:", PROJECT_ROOT)


## 1. 数据路径与运行配置

服务器上只需改 DATA_DIR，或指定 SOURCE_H5AD。默认先尝试项目 data 目录；本地自动识别用户提供的原始数据目录；两者都没有时尝试项目内合并文件。

X 是非负非整数，原文件未注明是否已经 log 变换。默认仅在分折后做 log1p(X)，不做额外文库大小归一化；expression_transform 控制训练与输出尺度；设为 none 表示训练保留源 X，主指标评价时仍按周报做 log1p(max(X,0))。如有证据表明源 X 已 log 化，需额外明确跨方法评价尺度。

主表达指标使用每个细胞的 `log1p(max(源 X, 0))`，然后在细胞维度求每个基因的平均值，对这 155 维向量计算 MSE、MAE、PCC。模型已在 log1p 空间输出时仅裁零，不重复 log。


In [ ]:
# 原始数据目录：服务器可改成 Path("/your/server/path/merfish_mouse_hypothalamus")。
DATA_DIR = Path(os.environ.get("MERFISH_DATA_DIR", str(PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus")))
LOCAL_DATA_DIR = Path("D:/Code/DeepSpatial/data/merfish_mouse_hypothalamus/merfish_mouse_hypothalamus-20261017T164627Z-1-001/merfish_mouse_hypothalamus")
if "MERFISH_DATA_DIR" not in os.environ and not list(DATA_DIR.glob("merfish_*.h5ad")) and LOCAL_DATA_DIR.is_dir():
    DATA_DIR = LOCAL_DATA_DIR

# 指定合并文件时直接使用它；None 表示优先读原始目录。
SOURCE_H5AD = None  # 例如 PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus_3d.h5ad"
SMOKE_TEST = False
EVALUATE_ONLY = False  # 已有 prediction.h5ad 时设 True，仅重算指标，不重新训练
RUN_NAME = "merfish_loso_log1p_predicted_annotation"
HELDOUT_SLICES = ["merfish_1", "merfish_2", "merfish_3"]

CONFIG = mh.default_config(smoke=SMOKE_TEST)
# 以下参数可微调；不同配置请使用新的 RUN_NAME。
CONFIG["device"] = "cuda:0"
CONFIG["expression_transform"] = "log1p"  # "log1p" / "none"
CONFIG["condition"] = "spatial_knn"       # "spatial_knn" / "none"（空间条件变体）
CONFIG["label_key"] = "annotation"       # 此数据 annotation 为区域；可显式改为 cell_class
CONFIG["xy_division"] = 20.0
CONFIG["cell_type_key"] = "cell_class"  # 稀有细胞评价标签，与扩散条件 annotation 分开
CONFIG["rare_train_fraction"] = 0.01   # 仅用本折训练集定义：占比严格小于 1%
CONFIG["coverage_radii_um"] = [25, 50, 100]  # 保留周报的 50/100 µm
CONFIG["xy_unit_um"] = 1.0              # 按周报将源 XY 视为 µm；可明确修改换算系数
CONFIG["z_division"] = 0.01              # 模型量化尺度，不假定原始 Z 的物理单位
# CONFIG["batch_size"] = 128
# CONFIG["sample_batch_size"] = 512
# CONFIG["pretrain_epochs"] = 200
# CONFIG["align_epochs"] = 300
# CONFIG["diffusion_epochs"] = 500
# CONFIG["generation_seeds"] = [2024, 2025, 2026]

OUTPUT_DIR = PROJECT_ROOT / "output" / (RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
print("原始数据目录:", DATA_DIR)
print("输出目录:", OUTPUT_DIR)
display(pd.Series(CONFIG, name="value"))


## 2. 直接加载原始数据，补齐三维格式

转换只保留源 X、共同基因和原始空间位置，不提前在五张切片上拟合标准化器。细胞名增加切片前缀，避免跨片重名。若从合并文件读取，要求它由本项目转换脚本生成。


In [ ]:
if SOURCE_H5AD is not None:
    adata = ad.read_h5ad(Path(SOURCE_H5AD))
elif list(DATA_DIR.glob("merfish_*.h5ad")):
    adata = mh.load_merfish(DATA_DIR)
elif "MERFISH_DATA_DIR" not in os.environ and (PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus_3d.h5ad").is_file():
    adata = ad.read_h5ad(PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus_3d.h5ad")
else:
    raise FileNotFoundError("请设置 DATA_DIR（包含五个 merfish_*.h5ad）或 SOURCE_H5AD（转换后的合并文件）")

assert adata.obsm["spatial"].shape == (adata.n_obs, 3)
assert adata.obs_names.is_unique and adata.var_names.is_unique
assert all(s in mh.slice_table(adata)["slice_ID"].astype(str).tolist()[1:-1] for s in HELDOUT_SLICES)
print(adata)
display(mh.slice_table(adata))
x = mh.dense(adata.X)
print("源 X: min/max =", float(x.min()), float(x.max()), "; 非整数比例 =", float(np.mean(np.abs(x - np.rint(x)) > 1e-5)))
del x


### 坐标检查

下图共用坐标轴和比例，只显示文件中已有的位置。可检查切片的方向和位置是否可比较；范围接近或轮廓重叠本身不能证明完成了生物学配准。这里不利用留出表达估计变换。


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 7))
for sid in mh.slice_table(adata)["slice_ID"].astype(str):
    xy = adata.obsm["spatial_xy"][adata.obs["slice_ID"].astype(str).to_numpy() == sid]
    ax.scatter(xy[:, 0], xy[:, 1], s=1, alpha=0.3, label=sid)
ax.set_aspect("equal")
ax.set_xlabel("Source X")
ax.set_ylabel("Source Y")
ax.legend(markerscale=5)
plt.show()


### 可选：本地导出一个可上传服务器的合并文件

导出文件保持源 X 不变，约含 28,317 × 155 的数据。服务器设置 SOURCE_H5AD 后可跳过读取五个文件。


In [ ]:
EXPORT_MERGED = False
if EXPORT_MERGED:
    merged_path = PROJECT_ROOT / "data" / "merfish_mouse_hypothalamus_3d.h5ad"
    merged_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(merged_path, compression="gzip")
    print("Saved:", merged_path)


## 3. 训练/查询隔离检查

每次留出后才处理训练表达、构图、拟合坐标原点和推断条件。query 的 X 全为零，obs 只包含源位置标识和**推断**条件，没有真实 annotation 或 cell_class。

保留查询坐标是本实验定义的一部分。对目标表达的评估在模型生成后执行。此处仅展示训练和查询规模，运行后可释放预览对象。


In [ ]:
for sid in HELDOUT_SLICES:
    train_preview, query_preview, truth_preview, info = mh.make_fold(adata, sid, CONFIG)
    assert not set(train_preview.obs_names).intersection(query_preview.obs_names)
    assert not np.any(query_preview.X)
    assert CONFIG["label_key"] not in query_preview.obs
    print(f"留出 {sid}: 训练 {train_preview.n_obs} 个细胞 / 查询 {query_preview.n_obs} 个细胞")
    print("  训练切片:", mh.slice_table(train_preview)["slice_ID"].tolist())
    print("  查询字段:", query_preview.obs.columns.tolist())
    print("  量化坐标范围:", query_preview.obsm["new_spatial"].min(0), query_preview.obsm["new_spatial"].max(0))
del train_preview, query_preview, truth_preview


## 4. 在服务器环境检查模型依赖

本单元格开始需要已跑通 T4 的 torch、PyG、diffusers 环境。155 基因的 MERFISH 与 2000 基因的果蝇模型输入不同，每折会重新初始化模型，不加载果蝇权重。

训练仍使用 T4 的图自编码器、跨片 triplet 对齐和三维条件扩散。新增的是分折处理与未知条件推断，细节在辅助脚本中。对齐阶段仍会处理整张训练切片，修改 batch_size 不会把该阶段改为小批量。


In [ ]:
if EVALUATE_ONLY:
    print("仅对已保存的平均预测重算周报指标，不导入训练模型，也不需要 GPU。")
else:
    import torch
    import diffusers
    import torch_geometric
    print("torch:", torch.__version__, "CUDA:", torch.version.cuda)
    print("diffusers:", diffusers.__version__, "PyG:", torch_geometric.__version__)
    if CONFIG["device"].startswith("cuda"):
        assert torch.cuda.is_available(), "当前内核无法使用 CUDA，请选择服务器训练环境"
        print("GPU:", torch.cuda.get_device_name(CONFIG["device"]))
    if SMOKE_TEST:
        print("SMOKE_TEST：每片 48 细胞、每阶段 1 轮、10 步生成；结果仅检查工程流程。")


## 5. 顺序运行三折实验

每折独立保存权重、归一化参数、划分/配置、环境版本、损失、逐次生成指标、平均预测和参考表达。这里只报告 STADiffuser。

- 相同数据和训练配置的已完成折会复用结果；旧预测没有周报指标时自动补算，不重训。只想重算设 EVALUATE_ONLY=True；此时只有已保存的平均预测可重算，不能恢复未保存的单次生成矩阵。
- 修改训练配置后换 RUN_NAME；未完成训练保留权重，本版本不自动恢复优化器状态。
- 默认三个生成结果平均，再计算主指标。生成重复不是训练重复。
- STADiffuser 本身没有 cell_class 输出；每折用四张训练切片的 log 表达和 cell_class 建立 1-NN 标注器，再标注生成表达。预测文件注明 cell_type_source，绝不把目标真实注释或空间邻居的真实类别直接复制到预测标签。稀有类结果依赖这一表达标注步骤，并非模型直接生成的分类标签。


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
frames = []
runner = mh.reevaluate_saved_fold if EVALUATE_ONLY else mh.run_fold
for sid in HELDOUT_SLICES:
    print("=" * 60, sid, flush=True)
    fold_metrics = runner(adata, sid, CONFIG, OUTPUT_DIR)
    frames.append(fold_metrics)
    display(fold_metrics[fold_metrics["seed"].astype(str) == "mean"])

all_metrics = pd.concat(frames, ignore_index=True)
all_metrics.to_csv(OUTPUT_DIR / "all_metrics.csv", index=False)
print("所选留出任务完成，输出：", OUTPUT_DIR)


## 6. 按周报汇总表达与稀有细胞指标

### 整体表达（主指标）

对真值和预测，先得到每个细胞的 log1p(max(源 X,0)) 表达，再在细胞维度取均值，形成 155 维基因向量；MSE、MAE 比较这两个向量，PCC 为其 Pearson 相关系数。MSE/MAE 越低越好，PCC 越高越好。不是逐元素或逐细胞表达误差；后者仅以 pointwise_* 作为辅助诊断。

### 稀有细胞与同类空间覆盖

本折四张训练切片中 cell_class 占比小于 1% 且训练中出现的类别定义为稀有类。对真实稀有细胞，检查 50/100 µm 内是否存在同类预测细胞，先算每类召回率，再对真实切片中出现的稀有类等权平均。整类没预测时召回为 0；没有预测点时精确率未定义。训练未见类单独标记，不混入训练稀有类。

同时输出逐类真值/预测数量、比例差、同类最近邻距离、类内平均表达 PCC 和覆盖精确率；cell_type_metrics.csv / rare_cell_metrics.csv 可追溯每类结果。

**已知坐标任务的边界：**目标细胞数量和 XY 坐标是输入，因此整体数量误差和不区分类别的几何距离固定为 0，不能用来宣称恢复了细胞数量或位置。稀有同类覆盖仍取决于生成表达推断出的类别。周报中 DeepSpatial 自由生成点集，任务输入不同，不能把这些固定值作公平的优劣排序。

summary 是三张留出切片的等权均值/样本标准差，训练集重叠，不解释为独立重复置信区间。development/smoke 结果不用于正式比较。


In [ ]:
fold_mean_metrics = all_metrics[all_metrics["seed"].astype(str) == "mean"].copy()
fold_mean_metrics.to_csv(OUTPUT_DIR / "fold_mean_metrics.csv", index=False)
metric_columns = ["mse", "mae", "pcc", "rare_recall_50um", "rare_recall_100um"]
summary = fold_mean_metrics.groupby(["method", "run_kind"], sort=False)[metric_columns].agg(["mean", "std"])
summary.to_csv(OUTPUT_DIR / "summary.csv")
display(fold_mean_metrics[["heldout", "method", "run_kind", *metric_columns,
                           "truth_n", "predicted_n", "missing_classes", "rare_missing_classes"]])
display(summary)

class_frames = []
for sid in HELDOUT_SLICES:
    table = pd.read_csv(OUTPUT_DIR / sid / "cell_type_metrics.csv")
    table.insert(0, "heldout", sid)
    class_frames.append(table)
all_classes = pd.concat(class_frames, ignore_index=True)
all_classes.to_csv(OUTPUT_DIR / "all_cell_type_metrics.csv", index=False)
all_rare = all_classes[all_classes["is_rare_train"] | all_classes["unseen_in_training"]].copy()
all_rare.to_csv(OUTPUT_DIR / "all_rare_cell_metrics.csv", index=False)
rare_present = all_rare[all_rare["is_rare_train"] & (all_rare["truth_n"] > 0)]
rare_summary = rare_present.groupby("cell_class").agg(
    truth_n=("truth_n", "sum"), predicted_n=("predicted_n", "sum"),
    recall_50um=("coverage_recall_50um", "mean"), recall_100um=("coverage_recall_100um", "mean"),
    expression_pcc=("mean_log_expression_pcc", "mean"))
rare_summary.to_csv(OUTPUT_DIR / "rare_class_summary.csv")
display(rare_summary)


## 7. 查看真实与预测的空间表达

同一目标坐标上对比真实/生成表达，每个基因在两列共用色标；这些是模型空间的表达值，指标表使用前述周报口径。真实标签和表达只在预测完成后参与评价/展示。


In [ ]:
PLOT_SLICE = HELDOUT_SLICES[0]
PLOT_GENES = ["Gad1", "Slc17a6", "Avp"]
fold_dir = OUTPUT_DIR / PLOT_SLICE
reference = ad.read_h5ad(fold_dir / "truth.h5ad")
predicted = ad.read_h5ad(fold_dir / "prediction.h5ad")
assert reference.obs_names.equals(predicted.obs_names) and reference.var_names.equals(predicted.var_names)
genes = [g for g in PLOT_GENES if g in reference.var_names]
genes += [g for g in reference.var_names if g not in genes][:max(0, 3 - len(genes))]
xy = reference.obsm["spatial_xy"]
fig, axes = plt.subplots(len(genes), 2, figsize=(9, 3.5 * len(genes)), squeeze=False, sharex=True, sharey=True)
for row, gene in enumerate(genes):
    values = [mh.dense(a[:, [gene]].X).ravel() for a in (reference, predicted)]
    low, high = min(v.min() for v in values), max(v.max() for v in values)
    for col, (name, value) in enumerate(zip(["Reference", "STADiffuser"], values)):
        ax = axes[row, col]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=value, s=3, cmap="viridis", vmin=low, vmax=high)
        ax.set_title(f"{gene} | {name}")
        ax.set_aspect("equal")
    fig.colorbar(sc, ax=axes[row].tolist(), label=CONFIG["expression_transform"])
fig.suptitle(f"Held-out {PLOT_SLICE}")
plt.show()


## 8. 查看稀有细胞数量与同类空间覆盖

下面展示每折每类的真值/预测数、50/100 µm 同类召回和类内表达 PCC。类名来自 cell_class；prediction 的类别来自训练表达 1-NN，不是真值继承字段。


In [ ]:
rare_columns = ["heldout", "cell_class", "train_n", "train_fraction", "truth_n", "predicted_n",
                "missing_in_prediction", "unseen_in_training", "coverage_recall_50um",
                "coverage_recall_100um", "truth_to_pred_nn_mean_um", "mean_log_expression_pcc"]
display(all_rare[rare_columns])


## 服务器后台运行

先将项目新增的 experiments 目录和本 Notebook 一起上传，在服务器配置路径后，从项目根目录、已激活的训练环境中运行：

```bash
tmux new -s merfish_loso
python -m jupyter nbconvert --to script docs/source/tutorials/T5-MERFISH-leave-one-slice-out.ipynb --output merfish_loso --output-dir .
python -u merfish_loso.py > merfish_loso.log 2>&1
```

Ctrl+B 后按 D 可断开会话，之后用 `tail -f merfish_loso.log` 查看日志。服务器没有图形界面时可使用 `MPLBACKEND=Agg python -u merfish_loso.py > merfish_loso.log 2>&1`。

[作者 T4 教程](https://stadiffuser.readthedocs.io/en/latest/tutorials/T4-3D-slice-modeling-Drosophila-embryo.html)提供模型结构与训练接口；本 Notebook 增加 MERFISH 数据适配和留出验证，不预填任何性能数值。
